# Experiment 2: stacking ensemble

Can a learned combiner improve on individual models and soft voting? The base learners are Logistic Regression, Random Forest, XGBoost, and Extra Trees. A regularized Logistic Regression meta-model learns from their **inner out-of-fold probabilities**.

No raw features are passed to the meta-model. The outer held-out participants are never used to train either layer. Base-model settings and meta-model C=0.1 are prespecified, not tuned from outer results.

**Status: written, not executed by the assistant.** Run cells in order. This notebook runs model evaluation, not software tests. CPU is sufficient. The voting notebook does not need to run first, although its results can be displayed at the end.

Because these experiments follow inspection of the initial results, treat them as exploratory rather than independent confirmation.

## 1. Locate the project

Use the same project files and original baseline artifacts as the voting notebook. Edit `PROJECT_ROOT` if needed. Drive is mounted only when you run this cell in Colab.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = None
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount('/content/drive')
    PROJECT_ROOT = PROJECT_ROOT or '/content/drive/MyDrive/ADNI_capstone'

if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p / 'src/scripts/train_models.py').is_file()), None)
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if ROOT is None or not (ROOT / 'src/scripts/ensemble_models.py').is_file():
    raise FileNotFoundError('Set PROJECT_ROOT to the updated project folder.')
print('Project:', ROOT)


## 2. Dependencies

Skip installation if this environment already has the modeling packages. If a kernel restart is requested, restart and rerun the project-path cell before continuing.

In [ ]:
requirements_file = str(ROOT / 'requirements-modeling.txt')
%pip install -r "{requirements_file}"


## 3. Freeze the settings

For each of the original five outer folds:

1. Use three inner folds within the outer training set to generate base-model out-of-fold probabilities.
2. Fit the regularized meta-model on those four probability columns and the training labels.
3. Refit each base pipeline on all outer training participants.
4. Predict the outer held-out participants with the base pipelines and pass those probabilities to the meta-model.
5. Save held-out predictions and evaluate at a fixed 0.5 threshold.

The meta-model never trains on base predictions made in-sample. Preprocessing is refitted within every inner training split. No early stopping or tuning uses outer labels.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown, Image

sys.path.insert(0, str(ROOT / 'src/scripts'))
from ensemble_models import load_inputs, run_experiment

cfg = json.loads((ROOT / 'config/ensembles.json').read_text())
METHOD = 'stacking'
REPORT = ROOT / 'reports/modeling' / cfg['run_name'] / METHOD
baseline_manifest, cohorts = load_inputs(cfg)
display(pd.DataFrame([{'experiment': name, 'participants': len(frame),
                       'converters': int(frame.label.sum()),
                       'nonconverters': int((frame.label == 0).sum())}
                      for name, frame in cohorts.items()]))
print(json.dumps(cfg, indent=2))


## 4. Train

This cell starts fitting. The four base models are evaluated alongside stacking. Existing run folders are not overwritten. To rerun, change `run_name` in the configuration before fitting. To inspect an already completed run, skip this cell.

In [ ]:
results = run_experiment(cfg, method=METHOD)


## 5. Performance and uncertainty

Accuracy measures correct predictions. Sensitivity measures detected converters, specificity measures correctly identified non-converters, and precision measures how often a positive prediction is right. F1 combines precision and sensitivity; balanced accuracy gives both classes equal weight. Higher AUROC/AP and lower Brier score are better.

Paired differences are stacking minus Logistic Regression on the same participants. Intervals spanning zero indicate uncertain improvement under this conditional bootstrap; they omit retraining uncertainty and are not multiplicity-adjusted.

In [ ]:
results = pd.read_csv(REPORT / 'metrics.csv')
columns = ['experiment', 'model', 'participants', 'accuracy', 'precision', 'sensitivity',
           'specificity', 'f1', 'balanced_accuracy', 'auroc', 'average_precision', 'brier_score']
formats = {c: '{:.1%}' for c in ['accuracy', 'precision', 'sensitivity', 'specificity', 'f1', 'balanced_accuracy']}
formats.update({c: '{:.3f}' for c in ['auroc', 'average_precision', 'brier_score']})
display(results[columns].style.format(formats))
display(results[['experiment', 'model', 'tn', 'fp', 'fn', 'tp']])
display(pd.read_csv(REPORT / 'paired_vs_logistic.csv').round(4))
display(pd.read_csv(REPORT / 'ensemble_parameters.csv').round(4))
display(Image(filename=str(REPORT / 'calibration_matched_fusion.png')))


## 6. Optional comparison with the voting run

This displays both sets of recorded results without retraining. They should share the same baseline data, settings, package versions, and outer folds. If you changed either run's settings or environment, document that before drawing a direct comparison. Selecting the best method here is exploratory model selection, not a new independent evaluation.

In [ ]:
voting_path = ROOT / 'reports/modeling' / cfg['run_name'] / 'voting/metrics.csv'
if voting_path.is_file():
    run_root = ROOT / 'artifacts/modeling' / cfg['run_name']
    voting_manifest = json.loads((run_root / 'voting/manifest.json').read_text())
    stacking_manifest = json.loads((run_root / 'stacking/manifest.json').read_text())
    for key in ['status', 'config', 'versions', 'baseline_manifest_sha256', 'script_sha256', 'shared_training_script_sha256']:
        if voting_manifest[key] != stacking_manifest[key]:
            raise ValueError(f'Runs differ in {key}; inspect them separately before comparing.')
    if voting_manifest['status'] != 'complete':
        raise ValueError('Both experiments must finish before combining results.')
    voting = pd.read_csv(voting_path)
    selected = voting[voting.model.isin(['equal_voting', 'weighted_voting'])]
    comparison = pd.concat([results, selected], ignore_index=True)
    display(comparison[columns].sort_values(['experiment', 'model']).style.format(formats))
else:
    print('No voting results found yet. Stacking results are available above.')
display(Markdown((REPORT / 'results.md').read_text()))


## Decision before deep learning

Record whether stacking improves on the simpler models, whether gains persist across folds, and whether calibration or sensitivity deteriorates. Keep all results, including negative findings. Do not keep adjusting methods against these same outer results to chase a target accuracy.

The matched cohort remains 222 ADNI1 participants, with a provisional endpoint and complete-case selection. MRI/CSF remain unsupported. Heavy architectures require a separate experiment design and a credible final evaluation strategy; this notebook does not establish that they will outperform the tabular models.